# Lab3 - Train Classifier (Spanish Only)

Classifier training setup:
- Train languages: ['es']
- Features: early fusion (XLM-R text + PySceneDetect keyframe stats + sensorial)
- Target: majority label from `labels_task3_1` (`YES`/`NO`)


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import joblib


In [ ]:
# Cluster configuration copied from run.sh / run_parallel.sh
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_SEQUENTIAL = 'shard:6'  # run.sh
SLURM_GRES_PARALLEL = 'shard:4'    # run_parallel.sh
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
PREDICCION_FINAL_DIR = PROJECT_ROOT / 'prediccion_final'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'

DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICCION_FINAL_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

FUSION_PATH = ARTIFACTS_DIR / 'fusion_features.npz'
META_PATH = ARTIFACTS_DIR / 'sample_metadata.csv'

GROUP_ID = 'BeingChillingWeWillWin'
MODEL_TAG = 'XLMRSceneSensor_LR_ES'
TEST_CASE = 'EXIST2026'
TRAIN_LANGS = ['es']

print('Training langs:', TRAIN_LANGS)
print('Using artifacts:', FUSION_PATH)
print('Final prediction dir:', PREDICCION_FINAL_DIR)

In [ ]:
if not FUSION_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError('Run 01_build_multimodal_features.ipynb first to generate artifacts.')

bundle = np.load(FUSION_PATH, allow_pickle=True)
X_all = bundle['X_fusion']
y_all = bundle['y']
langs_all = bundle['langs'].astype(str)
sample_ids_all = bundle['sample_ids'].astype(str)

meta = pd.read_csv(META_PATH)
meta['sample_id'] = meta['sample_id'].astype(str)

print('X_all shape:', X_all.shape)
print('Total samples:', len(sample_ids_all))
print('Language distribution:', dict(pd.Series(langs_all).value_counts()))


In [ ]:
train_mask = np.isin(langs_all, TRAIN_LANGS) & (y_all >= 0)

X_train_pool = X_all[train_mask]
y_train_pool = y_all[train_mask]

if len(np.unique(y_train_pool)) < 2:
    raise RuntimeError('Training subset has <2 classes. Check labels and language filter.')

print('Training subset size:', X_train_pool.shape[0])
print('Class distribution:', dict(pd.Series(y_train_pool).value_counts()))


In [ ]:
# Internal validation for quick quality check.
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_pool,
    y_train_pool,
    test_size=0.2,
    random_state=42,
    stratify=y_train_pool,
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])

clf.fit(X_tr, y_tr)
val_pred = clf.predict(X_va)

print('Validation macro-F1:', f1_score(y_va, val_pred, average='macro'))
print(classification_report(y_va, val_pred, digits=4))


In [ ]:
# Final fit on complete language-specific training subset.
final_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])
final_clf.fit(X_train_pool, y_train_pool)

meta = meta.sort_values('sample_id').reset_index(drop=True)
if not np.array_equal(sample_ids_all.astype(str), meta['sample_id'].astype(str).to_numpy()):
    raise RuntimeError('sample_metadata.csv is not aligned with fusion_features.npz by sample_id.')

split_series = meta['split'].fillna('').astype(str).str.upper() if 'split' in meta.columns else pd.Series([''] * len(meta))
official_test_mask = split_series.str.contains('TEST').to_numpy()

if not official_test_mask.any():
    raise RuntimeError(
        'No TEST rows found in sample_metadata.csv. Regenerate artifacts with official split before inference.'
    )

infer_mask = official_test_mask
infer_reason = 'official TEST split (strict)'

infer_ids = sample_ids_all[infer_mask]
X_infer = X_all[infer_mask]
pred_infer = final_clf.predict(X_infer)

label_map = {0: 'NO', 1: 'YES'}
pred_labels = [label_map[int(v)] for v in pred_infer]

print('Inference subset reason:', infer_reason)
print('Inference subset size:', int(infer_mask.sum()))

model_path = WEIGHTS_DIR / f'{GROUP_ID}_{MODEL_TAG}.joblib'
joblib.dump(final_clf, model_path)
print('Saved model:', model_path)

In [ ]:
output_rows = []
for sid, pred_label in zip(infer_ids, pred_labels):
    output_rows.append({
        'test_case': TEST_CASE,
        'id': str(sid),
        'value': pred_label,
    })

json_path = PREDICCION_FINAL_DIR / f'{GROUP_ID}_{MODEL_TAG}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output_rows, f, ensure_ascii=False)

print('Saved prediction json (prediccion_final):', json_path)
print('Rows:', len(output_rows))
output_rows[:3]

## Notes

- Inference is now strict: only rows where split contains TEST are used.
- Predictions are saved in prediccion_final/ under notebooks_shiyi.
- If TEST rows are missing in metadata, execution stops with an explicit error.